## Visualización de `datos_crypto_limpios.parquet`

Este notebook carga el dataset de precios cripto (índice temporal + columnas `*USDT`) y genera:
- Vista rápida del dataset
- Estadísticos y rango temporal
- Gráficas de precios (crudos y normalizados)
- Retornos (simples y log)
- Correlaciones
- Volatilidad rolling y drawdowns

**Archivo fuente**: `datos_crypto_limpios.parquet`


In [ ]:
# Si te falta alguna librería, descomenta y ejecuta:
# %pip install -U pandas pyarrow matplotlib seaborn


In [ ]:
from __future__ import annotations

import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_PATH = r"C:\Users\Usuario\Downloads\TAREA_PROCESADO_DATOS_FINANCIEROS_PARA_MACHINE_LEARNING\datos_crypto_limpios.parquet"
assert os.path.exists(DATA_PATH), f"No existe el archivo: {DATA_PATH}"

df = pd.read_parquet(DATA_PATH)

# Asegurar que el índice sea datetime (en este parquet ya viene como índice)
if not isinstance(df.index, pd.DatetimeIndex):
    # Si viniera como columna, intenta usarla
    for candidate in ["date", "datetime", "timestamp", "time", "__index_level_0__"]:
        if candidate in df.columns:
            df[candidate] = pd.to_datetime(df[candidate], errors="coerce")
            df = df.set_index(candidate)
            break
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors="coerce")

# Ordenar por tiempo
df = df.sort_index()

# Mantener solo columnas numéricas (precios)
price_cols = df.select_dtypes(include=["number"]).columns.tolist()
prices = df[price_cols].copy()

prices.head()

In [ ]:
# Vista general
print("Filas, columnas:", prices.shape)
print("Columnas:", list(prices.columns))
print("Rango temporal:", prices.index.min(), "->", prices.index.max())

# Frecuencia aproximada (mediana de diferencias)
deltas = prices.index.to_series().diff().dropna()
print("Frecuencia mediana:", deltas.median())

display(prices.tail())
display(prices.describe().T)

In [ ]:
# (Opcional) Submuestreo para que las gráficas sean más rápidas
# Si tus datos están cada 5 minutos, puedes bajar a 1h o 1D.
RESAMPLE_RULE = "1H"  # prueba "15min", "1H", "4H", "1D"
prices_rs = prices.resample(RESAMPLE_RULE).last().dropna(how="any")

print("Resample:", RESAMPLE_RULE, "->", prices_rs.shape)
prices_rs.head()

In [ ]:
# Gráfica de precios (múltiples series)
plt.figure(figsize=(14, 6))
for c in prices_rs.columns:
    plt.plot(prices_rs.index, prices_rs[c], linewidth=1, label=c)
plt.title(f"Precios (resample {RESAMPLE_RULE})")
plt.xlabel("Fecha")
plt.ylabel("Precio")
plt.legend(ncols=5, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Precios normalizados (base 100) para comparar rendimiento relativo
norm = (prices_rs / prices_rs.iloc[0]) * 100

plt.figure(figsize=(14, 6))
for c in norm.columns:
    plt.plot(norm.index, norm[c], linewidth=1, label=c)
plt.title(f"Precios normalizados (base 100) - resample {RESAMPLE_RULE}")
plt.xlabel("Fecha")
plt.ylabel("Índice (base 100)")
plt.legend(ncols=5, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Retornos (simples y log)
ret = prices_rs.pct_change().dropna(how="any")
logret = np.log(prices_rs).diff().dropna(how="any")

display(ret.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)

# Distribución de retornos (histogramas)
fig, axes = plt.subplots(2, 5, figsize=(16, 6), sharex=False, sharey=False)
axes = axes.ravel()
for i, c in enumerate(ret.columns):
    ax = axes[i]
    sns.histplot(ret[c], bins=100, kde=True, ax=ax)
    ax.set_title(c)
    ax.set_xlabel("ret")
plt.suptitle(f"Distribución de retornos simples - resample {RESAMPLE_RULE}")
plt.tight_layout()
plt.show()

In [ ]:
# Correlación (sobre retornos) + heatmap
corr = ret.corr()
plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title(f"Correlación de retornos - resample {RESAMPLE_RULE}")
plt.tight_layout()
plt.show()

# Pairplot (puede ser pesado; activa si quieres)
DO_PAIRPLOT = False
if DO_PAIRPLOT:
    sample = ret.sample(min(len(ret), 2000), random_state=42)
    sns.pairplot(sample)
    plt.show()

In [ ]:
# Volatilidad rolling (en retornos) y drawdown
WINDOW = 24 * 7  # si resample=1H -> 1 semana; ajusta según RESAMPLE_RULE

rolling_vol = ret.rolling(WINDOW).std() * np.sqrt(WINDOW)  # volatilidad anualizada aprox. (escala relativa)

plt.figure(figsize=(14, 6))
for c in rolling_vol.columns:
    plt.plot(rolling_vol.index, rolling_vol[c], linewidth=1, label=c)
plt.title(f"Volatilidad rolling (window={WINDOW}) - resample {RESAMPLE_RULE}")
plt.xlabel("Fecha")
plt.ylabel("Vol (aprox)")
plt.legend(ncols=5, fontsize=9)
plt.tight_layout()
plt.show()

# Drawdown sobre precios
cummax = prices_rs.cummax()
drawdown = prices_rs / cummax - 1

plt.figure(figsize=(14, 6))
for c in drawdown.columns:
    plt.plot(drawdown.index, drawdown[c], linewidth=1, label=c)
plt.title(f"Drawdown - resample {RESAMPLE_RULE}")
plt.xlabel("Fecha")
plt.ylabel("Drawdown")
plt.legend(ncols=5, fontsize=9)
plt.tight_layout()
plt.show()

display(drawdown.min().sort_values())  # máximo drawdown por activo

In [ ]:
# Dataset en formato largo (útil para algunas visualizaciones)
long_df = (
    prices_rs
    .reset_index(names=["datetime"])
    .melt(id_vars="datetime", var_name="symbol", value_name="price")
)

display(long_df.head())
print("long_df shape:", long_df.shape)

In [ ]:
# Ejemplo: comparar 2-3 activos en un periodo concreto
SYMS = ["BTCUSDT", "ETHUSDT", "BNBUSDT"]
START = "2021-01-01"
END = "2022-12-31"

sub = prices.loc[START:END, [s for s in SYMS if s in prices.columns]].resample(RESAMPLE_RULE).last().dropna(how="any")
sub_norm = (sub / sub.iloc[0]) * 100

plt.figure(figsize=(14, 5))
for c in sub_norm.columns:
    plt.plot(sub_norm.index, sub_norm[c], label=c, linewidth=1.5)
plt.title(f"Comparación normalizada ({START} -> {END}) - resample {RESAMPLE_RULE}")
plt.xlabel("Fecha")
plt.ylabel("Índice (base 100)")
plt.legend()
plt.tight_layout()
plt.show()